# Clinical Note data analysis (Snowflake, read-only)

One row = one clinical note. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows Table / Chart / Pivot and the **download** button.

**Grain:** `NoteId` should be unique. One patient / encounter can have many notes.

**Why this table is different**
- `NoteType` is a **taxonomy** (Progress, Consultation, Procedure, Discharge, Operative, Imaging, Laboratory, Pathology, Mental health, Other) plus many subtype examples. The vendor may store the high-level type, a subtype, or a mixed string. We **group by the stored string** — we do not guess the 10 buckets unless the text matches.
- `Clinical Note Text` is **full provider prose**. We never glue all notes into one cell (`LISTAGG` of full notes would fail and dump PHI). We count unique texts, find **duplicate** texts, measure length, and only preview the first 200 characters.

**Most important views**
1. Unique `NoteType` values with occurrences.
2. Unique `Clinical Note Text` values with occurrences (templates / copy-paste show up here).
3. Duplicate notes (same full text used more than once).
4. Note length and date spread.
5. Notes per patient and per encounter.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

`Clinical Note Text` has spaces, so it must be quoted. Keep `QUOTE_COLUMNS = True` unless `DESCRIBE` shows a simple uppercase name.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "CLINICAL_NOTE"   # try CLINICAL_NOTES, NOTE, NOTES, CLINICALNOTE

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "note_id": "NoteId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "note_type": "NoteType",
    "note_text": "Clinical Note Text",
    "date": "Date",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_ID = col("note_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_TYPE = col("note_type")
C_TEXT = col("note_text")
C_DATE = col("date")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_ID", C_ID),
    ("C_ENC", C_ENC),
    ("C_PT", C_PT),
    ("C_TYPE", C_TYPE),
    ("C_TEXT", C_TEXT),
    ("C_DATE", C_DATE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%NOTE%'
     OR UPPER(TABLE_NAME) LIKE '%CLINICAL%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

The preview cell shows only the **first 200 characters** of each note so the grid stays usable. Full text is still in the table; later cells count the full string.

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT
    {{C_ID}} AS NOTE_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_PT}} AS PATIENT_ID,
    {{C_TYPE}} AS NOTE_TYPE,
    LEFT({{C_TEXT}}::STRING, 200) AS NOTE_TEXT_PREVIEW,
    LENGTH({{C_TEXT}}::STRING) AS NOTE_CHAR_LENGTH,
    {{C_DATE}} AS NOTE_DATE
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

Expected: unique `NoteId` close to row count. Unique note **texts** will often be smaller than row count if templates or copy-paste notes repeat.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_ID}}) AS UNIQUE_NOTE_IDS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_TYPE}}) AS UNIQUE_NOTE_TYPES,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_NOTE_TEXTS,
    COUNT(*) - COUNT(DISTINCT {{C_ID}}) AS EXTRA_ROWS_VS_UNIQUE_NOTE_ID,
    COUNT(*) - COUNT(DISTINCT {{C_TEXT}}) AS EXTRA_ROWS_VS_UNIQUE_TEXT,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_NOTES_PER_PATIENT,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_ENC}}), 0), 2) AS AVG_NOTES_PER_ENCOUNTER
FROM {{T}};

## 6. Completeness (nulls)

All six dictionary fields are required. Blank or whitespace-only note text is counted separately from SQL null.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_ID}} IS NULL, 1, 0)) AS NULL_NOTE_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_TYPE}} IS NULL, 1, 0)) AS NULL_NOTE_TYPE,
    SUM(IFF({{C_TEXT}} IS NULL, 1, 0)) AS NULL_NOTE_TEXT,
    SUM(IFF({{C_TEXT}} IS NOT NULL AND TRIM({{C_TEXT}}::STRING) = '', 1, 0)) AS BLANK_NOTE_TEXT,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS NULL_DATE
FROM {{T}};

## 7. NoteType — unique values with occurrences (taxonomy check)

Dictionary buckets (not hardcoded in SQL):
1. Progress notes
2. Consultation notes
3. Procedure notes
4. Discharge summary
5. Operative notes
6. Imaging reports
7. Laboratory notes
8. Pathology notes
9. Mental health notes
10. Other notes

Vendors often store a subtype instead (`Cardiology consultation`, `X-ray findings`). `note_type_counts` lists **every stored string**. `note_type_mapped_to_bucket` is a **best-effort** map using keywords so you can see how close the data is to the 10 buckets. Unmapped values stay as `Unmapped - see NoteType as stored`.

In [ ]:
SELECT
    {{C_TYPE}} AS NOTE_TYPE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_NOTE_TEXTS,
    ROUND(AVG(LENGTH({{C_TEXT}}::STRING)), 0) AS AVG_CHAR_LENGTH,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC, NOTE_TYPE;

In [ ]:
WITH mapped AS (
    SELECT
        {{C_TYPE}} AS NOTE_TYPE,
        {{C_PT}} AS PATIENT_ID,
        CASE
            WHEN {{C_TYPE}} IS NULL THEN 'Unknown (missing NoteType)'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'progress|follow[- ]?up|post[- ]?surg|medication adjust', 'i') THEN '1. Progress notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'consult|referral|cardiology|neurology|oncology', 'i') THEN '2. Consultation notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'procedure note|endoscop|biopsy report|surgery summary', 'i') THEN '3. Procedure notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'discharge', 'i') THEN '4. Discharge summary'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'operative|preoperative|postoperative', 'i') THEN '5. Operative notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'imaging|radiolog|x[- ]?ray|mri|ultrasound', 'i') THEN '6. Imaging reports'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'laborator|lab note|blood test|microbiology|biochemical', 'i') THEN '7. Laboratory notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'patholog|histopath|cytolog|tissue sample', 'i') THEN '8. Pathology notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'social worker|nutrition|physical therap', 'i') THEN '10. Other notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'mental|psychiatr|therapy|cbt|suicide', 'i') THEN '9. Mental health notes'
            WHEN REGEXP_LIKE({{C_TYPE}}::STRING, 'other', 'i') THEN '10. Other notes'
            ELSE 'Unmapped - see NoteType as stored'
        END AS NOTE_TYPE_BUCKET
    FROM {{T}}
)
SELECT
    NOTE_TYPE_BUCKET,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT NOTE_TYPE) AS DISTINCT_STORED_NOTE_TYPES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM mapped
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    {{C_TYPE}} AS NOTE_TYPE,
    COUNT(*) AS ROW_COUNT,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
WHERE {{C_TYPE}} IS NOT NULL
  AND NOT REGEXP_LIKE(
        {{C_TYPE}}::STRING,
        'progress|follow[- ]?up|post[- ]?surg|medication adjust|consult|referral|cardiology|neurology|oncology|procedure note|endoscop|biopsy report|surgery summary|discharge|operative|preoperative|postoperative|imaging|radiolog|x[- ]?ray|mri|ultrasound|laborator|lab note|blood test|microbiology|biochemical|patholog|histopath|cytolog|tissue sample|mental|psychiatr|therapy|cbt|suicide|other|social worker|nutrition|physical therap',
        'i'
      )
GROUP BY 1
ORDER BY ROW_COUNT DESC;

## 8. Clinical Note Text — unique texts, occurrences, duplicates (most important)

We group by the **full stored string**. Two notes that differ by one character are two unique texts.

- `note_text_summary` — how many unique texts vs rows.
- `note_text_duplicate_list` — texts used **more than once** (templates, copied phrases). Preview is 300 characters; `ROW_COUNT` is the occurrence.
- `note_text_unique_with_occurrences` — **every** unique text with its occurrence. This can be a very large download. Prefer the duplicate list first if the table is huge.
- Length bands tell you empty vs short template vs long narrative.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_NOTE_TEXTS,
    COUNT(DISTINCT TRIM({{C_TEXT}}::STRING)) AS UNIQUE_TEXTS_TRIMMED,
    SUM(IFF({{C_TEXT}} IS NULL OR TRIM({{C_TEXT}}::STRING) = '', 1, 0)) AS EMPTY_OR_NULL_TEXT_ROWS,
    COUNT(*) - COUNT(DISTINCT {{C_TEXT}}) AS REPEAT_ROWS_BEYOND_UNIQUE_TEXT,
    MIN(LENGTH({{C_TEXT}}::STRING)) AS MIN_CHAR_LENGTH,
    ROUND(AVG(LENGTH({{C_TEXT}}::STRING)), 0) AS AVG_CHAR_LENGTH,
    MEDIAN(LENGTH({{C_TEXT}}::STRING)) AS MEDIAN_CHAR_LENGTH,
    MAX(LENGTH({{C_TEXT}}::STRING)) AS MAX_CHAR_LENGTH
FROM {{T}};

In [ ]:
SELECT
    CASE
        WHEN {{C_TEXT}} IS NULL THEN 'Unknown (null)'
        WHEN TRIM({{C_TEXT}}::STRING) = '' THEN 'Blank'
        WHEN LENGTH({{C_TEXT}}::STRING) <= 50 THEN '1-50 characters'
        WHEN LENGTH({{C_TEXT}}::STRING) <= 200 THEN '51-200 characters'
        WHEN LENGTH({{C_TEXT}}::STRING) <= 500 THEN '201-500 characters'
        WHEN LENGTH({{C_TEXT}}::STRING) <= 2000 THEN '501-2000 characters'
        WHEN LENGTH({{C_TEXT}}::STRING) <= 8000 THEN '2001-8000 characters'
        ELSE 'More than 8000 characters'
    END AS LENGTH_BAND,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_TEXTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    LEFT({{C_TEXT}}::STRING, 300) AS NOTE_TEXT_PREVIEW,
    LENGTH({{C_TEXT}}::STRING) AS NOTE_CHAR_LENGTH,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_TYPE}}) AS DISTINCT_NOTE_TYPES,
    LISTAGG(DISTINCT {{C_TYPE}}::STRING, ' | ') AS NOTE_TYPES,
    MIN({{C_DATE}}) AS FIRST_DATE,
    MAX({{C_DATE}}) AS LAST_DATE
FROM {{T}}
WHERE {{C_TEXT}} IS NOT NULL
GROUP BY {{C_TEXT}}
HAVING COUNT(*) > 1
ORDER BY ROW_COUNT DESC;

In [ ]:
SELECT
    LEFT({{C_TEXT}}::STRING, 300) AS NOTE_TEXT_PREVIEW,
    LENGTH({{C_TEXT}}::STRING) AS NOTE_CHAR_LENGTH,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_TYPE}}) AS DISTINCT_NOTE_TYPES,
    LISTAGG(DISTINCT {{C_TYPE}}::STRING, ' | ') AS NOTE_TYPES
FROM {{T}}
GROUP BY {{C_TEXT}}
ORDER BY ROW_COUNT DESC;

### NoteType × text uniqueness

For each note type: how many notes, how many distinct texts, and how repetitive the type is (`REPEAT_RATIO` close to 1 means almost every note reuses the same text).

In [ ]:
SELECT
    {{C_TYPE}} AS NOTE_TYPE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_TEXTS,
    ROUND(1 - (COUNT(DISTINCT {{C_TEXT}}) / NULLIF(COUNT(*), 0)), 4) AS REPEAT_RATIO,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_TEXT}}), 0), 2) AS AVG_USES_PER_UNIQUE_TEXT
FROM {{T}}
GROUP BY 1
ORDER BY REPEAT_RATIO DESC, ROW_COUNT DESC;

### Word counts inside Clinical Note Text (use after the duplicate list)

This splits **full notes** into words. It is heavier than other cells. `WORD_OCCURRENCES` is how many times that word appears across all notes (a long note that repeats a word counts more than once).

Common English words (`the`, `and`, `patient`) will dominate. That is expected; scan past them for clinical terms.

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_TEXT}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_TEXT}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
  AND LENGTH(WORD) >= 3
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD
LIMIT 500;

## 9. Note Date distribution

`Date` is when the note was **recorded**. Recency is today minus that date.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DATE}}) AS MIN_DATE,
    MAX({{C_DATE}}) AS MAX_DATE,
    DATEDIFF('day', MIN({{C_DATE}})::DATE, MAX({{C_DATE}})::DATE) AS SPAN_DAYS,
    SUM(IFF({{C_DATE}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_ROWS,
    SUM(IFF({{C_DATE}} IS NULL, 1, 0)) AS MISSING_DATE_ROWS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS NOTE_YEAR,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_TYPE}}) AS UNIQUE_NOTE_TYPES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    YEAR({{C_DATE}}) AS NOTE_YEAR,
    {{C_TYPE}} AS NOTE_TYPE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS
FROM {{T}}
GROUP BY 1, 2
ORDER BY 1, ROW_COUNT DESC;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DATE}} IS NULL THEN 90
            WHEN {{C_DATE}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DATE}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DATE}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 10. Notes per patient and per encounter

In [ ]:
WITH per_patient AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS NOTE_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    NOTE_COUNT AS NOTES_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM per_patient
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH per_enc AS (
    SELECT
        {{C_ENC}} AS ENCOUNTER_ID,
        COUNT(*) AS NOTE_COUNT
    FROM {{T}}
    WHERE {{C_ENC}} IS NOT NULL
    GROUP BY 1
)
SELECT
    NOTE_COUNT AS NOTES_ON_ENCOUNTER,
    COUNT(*) AS NUMBER_OF_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ENCOUNTERS
FROM per_enc
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS NOTE_COUNT,
    COUNT(DISTINCT {{C_TYPE}}) AS UNIQUE_NOTE_TYPES,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_TEXT}}) AS UNIQUE_TEXTS,
    MIN({{C_DATE}}) AS FIRST_DATE,
    MAX({{C_DATE}}) AS LAST_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY NOTE_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ID}} AS NOTE_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_TYPE}} AS NOTE_TYPE,
    {{C_DATE}} AS NOTE_DATE,
    LENGTH({{C_TEXT}}::STRING) AS NOTE_CHAR_LENGTH,
    LEFT({{C_TEXT}}::STRING, 300) AS NOTE_TEXT_PREVIEW
FROM {{T}}
WHERE {{C_PT}} = (
        SELECT {{C_PT}}
        FROM {{T}}
        WHERE {{C_PT}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DATE}} NULLS LAST, {{C_ID}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid.
- Full clinical note text is **PHI**. Previews are truncated. Do not `LISTAGG` full notes.
- `note_text_unique_with_occurrences` can return one row per distinct note. Start with `note_text_duplicate_list` if the table is large.
- Word-count cell is limited to the top 500 words of length 3+.
- Run `config` before the SQL cells.